# Continuation Mechanical Systems with High Damping
### Saran Kaba (sk65556)

### The equation being used is
$\frac{dx}{dt}=-\frac{k}{c}+\frac{1}{c}F(t)$
#### in which the stiffness parameters are
$damping-rate:\frac{k}{c}=5000s^-1$  
$external-force:F(t)=10sin(t) N$   
$initial-conditions:x(0)=1m$
#### The damping rate describes how fast the oscillations are being depressed, while the external force in the force being exterted on the system. The initial conditions shows the initial displacement to be at 1 meter. Having an extremely high damping rate while having a relatively slow force defines the system as a stiff equation. Understanding this system is valuable in machanical structures as it allows for the improvement of stability and function.

## Newton-Raphson Damping

### While Euler's Forward showed its ability to exhibit stability even with large step sizes, there is still the possibility for the method to calcualted an answer with a high error when the step size is too large. In comes the method of Newtom-Raphson with damping and line search. The damping coefficient, between zero and one, is used to scale down the step size. The ideal damping is found through an algorithm of line searching. It starts of with a size of one, which is the full step, and gradually decreases until convergence is reached.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
def analytical_solution(t):
    return np.exp(-5000*t)+4e-4*np.sin(t)-8e-8*np.cos(t)
def f(x, y):
    return -5000*y + 0.002*np.sin(x)
def fprime(y):
    return -5000

# Newton-Raphson for single step with Backward Euler
def backward_euler_step(f, y, t, h, tol=1e-6, max_iter=20):
    total_iterations=0  #calculating the number of iterations per time step
    y_next = y  # Initial guess
    for _ in range(max_iter):
        total_iterations+=1
        g_val = y_next - y - h * f(t + h, y_next)
        g_prime_val = 1 - h * fprime(y_next)     
        if abs(g_prime_val) < tol:  # Avoid division by zero
            break
        y_new = y_next - g_val / g_prime_val
        if abs(y_new - y_next) < tol:
            return y_new,total_iterations  # Converged
        y_next = y_new
    return y_next,total_iterations




# No Damping Newton Raphson
lam=5000.
def analytical_solution(t):
    return np.exp(-5000*t)+4e-4*np.sin(t)-8e-8*np.cos(t)

def residual(y_next, y_n, t_next, h):
    """g(y_{n+1}) = y_{n+1} - y_n - h*f(t_{n+1}, y_{n+1})"""
    return y_next - y_n - h * f(t_next, y_next)

def residual_derivative(h):
    """g'(y_{n+1}) = 1 - h*(df/dy) = 1 + lam*h"""
    return 1 + lam * h

def newton_raphson_step(y_n, t_next, h, tol, max_iter=20):
    """Solve for y_{n+1} using Newton-Raphson"""
    y_guess = y_n
    g_prime = residual_derivative(h)
    error=0
    
    for k in range(max_iter):
        g = residual(y_guess, y_n, t_next, h)
        if abs(g) < tol:
            return y_guess, k,error
        delta = -g / g_prime
        y_guess = y_guess + delta
        error=np.abs(analytical_solution(t_next)-y_guess)
    
    return y_guess, max_iter,error

## Adaptive Step-Size Backward Euler

In [2]:
# Adaptive Backward Euler
def backward_euler_adaptive(f, y0, t0, tf, h_init, tol=1e-6, max_iter=20):
    t_values = [t0]
    y_values = [y0]
    h = h_init
    t = t0
    y = y0
    tot_it_f=[0]
    sol_error=[np.abs(analytical_solution(t0)-y0)]
    h_accepted=0
    h_rejected=0
    h_used=[]
    while t < tf:
        if t + h > tf:  # Adjust step size to not overshoot
            h = tf - t
        # Single step with h
        y_full,t_it_f = backward_euler_step(f, y, t, h, tol, max_iter)
        # Two half-steps with h/2
        h_half = h / 2
        y_half_1,t_it_1 = backward_euler_step(f, y, t, h_half, tol, max_iter)
        y_half_2,t_it_2 = backward_euler_step(f, y_half_1, t + h_half, h_half, tol, max_iter)
        # Error estimation
        error = abs(y_full - y_half_2)
        # Adjust step size
        if error > tol:
            h /= 2  # Decrease step size
            h_rejected +=1
            continue  # Retry the step with smaller h
        elif error < tol / 2:
            h *= 2  # Increase step size for efficiency
        # Accept the step
        h_accepted+=1
        h_used.append(h)
        t += h
        y = y_half_2
        t_values.append(t)
        y_values.append(y)
        tot_it_f.append(t_it_f+t_it_1+t_it_2)
        #Calculating the error
        sol_error.append(np.abs(analytical_solution(t)-y))

    return np.array(t_values), np.array(y_values),np.array(tot_it_f),np.array(sol_error),np.array(h_accepted),np.array(h_rejected),np.array(h_used)

#Finding the analytical solution
analytical= analytical_solution(t_values)

NameError: name 't_values' is not defined

In [ ]:
# Defining Values
y=1
t0=0
tf=2.0
h=0.1

#With damping
t_values,y_values,total_iterations,error,t_acc,t_rej,h_u=backward_euler_adaptive(f, y, t0, tf, h, tol=1e-6, max_iter=20)

#Without Damping
max_it_nodamp=[0]
y_nodamp=[y]
error_no=[np.abs(analytical_solution(t0) - y)]
for i in range (len(t_values)-1):
    yno, mait,err=newton_raphson_step(y_nodamp[-1], t_values[i+1], t_values[i+1]-t_values[i], tol=1e-6, max_iter=20)
    max_it_nodamp.append(mait)
    y_nodamp.append(yno)
    error_no.append(err)
max_it_nodamp=np.array(max_it_nodamp)
y_nodamp=np.array(y_nodamp)
error_no=np.array(error_no)


plt.figure()
plt.plot(t_values,analytical,label='Analytical Solution')
plt.plot(t_values,y_nodamp,label='Backward Euler Without Damping')
plt.plot(t_values,y_values,label='Adaptive Backward Euler')
plt.xlabel('time step')
plt.ylabel('position')
plt.xscale('log')
plt.yscale('log')
plt.grid(True)
plt.legend()
plt.title('Adaptive Backwards Euler')
plt.show()

In [ ]:
#Number of iterations
plt.figure()
plt.plot(t_values,max_it_nodamp,label='Convergence of Damped NR Behavior')
plt.plot(t_values,total_iterations,label='Convergence behavior with damping')
plt.title('Comparing number of iterations')
plt.xlabel('Time Step')
plt.ylabel('Number of Iterations')
plt.tight_layout()
plt.xscale('log')
plt.grid(True)
plt.legend()
plt.show()

### Note the higher number of iterations for the damping vs no damping method. This is due to the damping method using multiples of the same h value before figuring which fits better. While this method can help find the more effiecient step size, it decreasing the efficiency of the whole system by increasing the computational burden and time it takes to solve. This may be offset by the potential for a smaller error:

In [ ]:
plt.figure()
plt.plot(t_values,error_no,label='No Damping Error')
plt.plot(t_values,error,label='Newton-Raphson with Backward Euler Error')
plt.xlabel('Time Step')
plt.ylabel('Error')
plt.title('Comparing Error')
plt.xscale('log')
plt.yscale('log')
plt.legend()
plt.tight_layout()
plt.show()

### From the data gathered, it can be concluded the no damping method actually performed better in calculating error, even between the time steps of 0.00001 and 0.001, though with dampng showed a more consistent error calculation beyond that.

## Conclusion

### To conclude, it was found that without dampining seemed to perform better when reaching the balance between accuracy and computational stress. While with damping showed a more consistent error analysis, it was not only larger but did not offset the time taken and computational cost to find the solution with the given parameters. All in all, Adaptive Backward Euler failed in these trials to show a superior solver in the given scenario